# Week 4: Predict & Explain — Regression on Real DASS Data

Welcome to your first machine learning modelling challenge! You'll build regression models to predict depression scores from personality traits and demographics, using real data from nearly 40,000 people.

**In this lab, you'll use the vocabulary from Week 3** — features, targets, train/test splits, cross-validation, regularisation — on a real dataset.

**Remember the LLM Problem-Solving Loop:**

**Outer loop (your research process):** PLAN → EXECUTE → EVALUATE → DOCUMENT

**Inner loop (working with the AI):** ENGINEER → PLAN → GENERATE → VERIFY → REFINE

**This week's new skill: debugging.** When code breaks, share the full error with your AI assistant and ask it to explain what went wrong — not just fix it.

See the [challenge brief (README.md)](README.md) for full details.

---

In [ ]:
# === IMPORTS ===
# These are the libraries you'll use today.
# pandas: for loading and working with data tables
# numpy: for numerical operations
# matplotlib + seaborn: for creating plots
# sklearn: for building and evaluating regression models

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

# Set some defaults so our plots look nice
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded successfully!")

In [ ]:
# === LOAD THE DATA ===
# This file is TAB-separated (not comma-separated), so we need sep="\t"
# If you forget this, you'll get a DataFrame with 1 column — a common gotcha!

data = pd.read_csv("data/dass42_data.csv", sep="\t")

# Always check shape and head right after loading
print(f"Dataset shape: {data.shape[0]} respondents, {data.shape[1]} variables")
print()
data.head()

## Step 1: Quick Data Review

Before doing anything else, let's understand what we're working with. This dataset has 172 columns — that's a lot! The key variable groups are:

- **Q1A – Q42A**: DASS-42 item responses (coded 1–4)
- **TIPI1 – TIPI10**: Big Five personality items (coded 1–7)
- **Demographics**: age, gender, education, urban, married, familysize, etc.
- **VCL1 – VCL16**: Vocabulary check (VCL6, VCL9, VCL12 are fake words)
- **Q1E – Q42E**: Item response times (milliseconds) — careful, these are a trap!

Run the cell below to get summary statistics and check for missing values.

In [ ]:
# Summary statistics for key columns
key_cols = ["age", "gender", "education", "TIPI1", "TIPI2", "TIPI3", "TIPI4", "TIPI5",
            "Q3A", "Q5A", "Q10A"]  # a few DASS items
print("Summary statistics for key variables:")
print(data[key_cols].describe().round(2))
print()

# Check for missing values in DASS items and TIPI items
dass_cols = [f"Q{i}A" for i in range(1, 43)]
tipi_cols = [f"TIPI{i}" for i in range(1, 11)]
print(f"Missing values in DASS items: {data[dass_cols].isnull().sum().sum()}")
print(f"Missing values in TIPI items: {data[tipi_cols].isnull().sum().sum()}")
print()

# Check the age distribution — are there any clearly wrong values?
print(f"Age range: {data['age'].min()} to {data['age'].max()}")
print(f"Ages > 100: {(data['age'] > 100).sum()}")

## Step 2: Basic Cleaning

The data has some quality issues we should fix before modelling:

1. **Bad ages** — some respondents entered their birth year (e.g., 1998) instead of their age. We'll keep only ages 10–100.
2. **Careless responders** — the vocabulary check includes three fake words (VCL6, VCL9, VCL12). Anyone who claims to know 2 or more fake words was probably not paying attention.

Run the cell below to clean the data.

In [ ]:
# Filter unreasonable ages
print(f"Before cleaning: {len(data)} rows")

data = data[(data["age"] >= 10) & (data["age"] <= 100)]
print(f"After age filter (10-100): {len(data)} rows")

# Remove careless responders (endorsed 2+ fake vocabulary words)
fake_words = ["VCL6", "VCL9", "VCL12"]
data["vcl_fake_count"] = data[fake_words].sum(axis=1)
n_careless = (data["vcl_fake_count"] >= 2).sum()
data = data[data["vcl_fake_count"] < 2]
print(f"Removed {n_careless} careless responders")
print(f"Clean dataset: {len(data)} rows")

## Step 3: PLAN Your Analysis

Before writing any modelling code, ask your AI assistant to create an analysis plan. This is the **PLAN** step of the outer loop — and it's the most important one.

**Why plan first?** A written plan catches misunderstandings before code is written. It's much easier to say "actually, let's try different features" when you're looking at a plan than after the AI has written 100 lines of modelling code.

### What to do

1. Open your AI assistant (Copilot Chat, ChatGPT, Claude, etc.)
2. Use this prompt (or adapt it):

> "I have a pandas DataFrame called `data` loaded from a tab-separated file with 39,775 rows and 172 columns. This is real survey data from the Depression Anxiety Stress Scales (DASS-42). The key variables are: 42 DASS items (Q1A–Q42A, coded 1–4), which need recoding to 0–3 and summing into Depression, Anxiety, and Stress subscales. The target is DASS Depression (14 items, total 0–42). Potential features include: Big Five personality (TIPI1–TIPI10), demographics (age, gender, education, urban, married, familysize, orientation, race), and vocabulary check items (VCL1–VCL16, where VCL6, VCL9, VCL12 are fake words for validity screening). Create a plan for building and comparing regression models (linear regression, Ridge, and Lasso) to predict depression scores from personality and demographics. Include: how to score the DASS subscales, which features to use and why, how to handle data quality, how to split the data, how to evaluate the models, and how to compare them. Don't write code yet — just the plan."

3. **Paste the AI's plan in the cell below.** Review it, revise if needed.

### Your AI-Generated Plan

*Paste your AI's analysis plan here. Double-click this cell to edit it.*

*Once you've reviewed and revised the plan, move on to Step 4.*

## Step 4: Score the DASS and TIPI

The raw data has individual item responses, but we need computed scores:

**DASS Depression subscale:**
- 14 items: Q3, Q5, Q10, Q13, Q16, Q17, Q21, Q24, Q26, Q31, Q34, Q37, Q38, Q42
- Raw items are coded 1–4 → recode to 0–3 (subtract 1) → sum
- Final score: 0–42

**Big Five personality (TIPI):**
- 10 items, rated 1–7
- Items 2, 4, 6, 8, 10 need **reverse scoring** (reverse = 8 − original)
- Each trait = average of its two items

Ask your AI to help write this code. Here's the item information to include in your prompt:
- Depression items: Q3, Q5, Q10, Q13, Q16, Q17, Q21, Q24, Q26, Q31, Q34, Q37, Q38, Q42
- TIPI reverse items: 2, 4, 6, 8, 10
- Extraversion = (TIPI1 + TIPI6R) / 2, Agreeableness = (TIPI2R + TIPI7) / 2, etc.

In [ ]:
# Ask your AI to help score the DASS Depression subscale and Big Five traits.
# Paste the code here and run it.
#
# Hints:
#   dep_items = [f'Q{i}A' for i in [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]]
#   data['DASS_Depression'] = data[dep_items].sub(1).sum(axis=1)
#
#   For TIPI reverse scoring: data['TIPI2R'] = 8 - data['TIPI2']
#   Then: data['Extraversion'] = (data['TIPI1'] + data['TIPI6R']) / 2



## Step 5: Choose Features and Target

The **target** (what we're trying to predict) is `DASS_Depression`.

The **features** (what we're using to predict) are up to you. A good starting point is the Big Five personality traits — these are the theoretically motivated predictors. You can add demographics later to see if they help.

**Don't include** DASS Anxiety or Stress as features — that would be circular (they're from the same questionnaire). And be cautious about including response time variables (Q1E, Q2E, etc.) — they can inflate R² without adding psychological meaning.

In [ ]:
# Define your target and features
# target = "DASS_Depression"
# features = ["Extraversion", "Agreeableness", "Conscientiousness", "EmotionalStability", "Openness"]
# 
# # Optional: add demographics
# # features += ["age", "gender", "education"]
# 
# # Prepare X and y
# model_data = data[features + [target]].dropna()
# X = model_data[features]
# y = model_data[target]
# print(f"Model data: {X.shape[0]} rows, {X.shape[1]} features")



## Step 6: Train/Test Split

Remember from Week 3: we split the data into a **training set** (for building and tuning models) and a **test set** (for final evaluation only). The test set stays locked until Step 11.

Use `random_state=42` so your results are reproducible — everyone in the class will get the same split.

In [ ]:
# Split into training (80%) and test (20%)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# print(f"Training set: {len(X_train)} rows")
# print(f"Test set: {len(X_test)} rows (locked until Step 11!)")



## Step 7: Baseline Model — What's the Dumbest Prediction?

Before building any fancy models, establish a **baseline**. The simplest possible prediction: just guess the mean depression score for everyone. Any real model should beat this.

Calculate the baseline MAE (Mean Absolute Error) and R². These are the numbers to beat.

In [ ]:
# Baseline: predict the mean for everyone
# Ask your AI: "How do I calculate baseline MAE and R² when predicting the mean?"
#
# Hint: you can use DummyRegressor(strategy='mean') from sklearn
# or just compute it manually: mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))



## Step 8: Build Models — Linear, Ridge, Lasso

Now build three regression models and evaluate each with **5-fold cross-validation** on the training set:

1. **Linear Regression** (ordinary least squares) — the straightforward starting point
2. **Ridge** (L2 regularisation, alpha=1.0) — shrinks coefficients, handles collinearity
3. **Lasso** (L1 regularisation, alpha=0.1) — can drop features entirely

Use `cross_val_score` from sklearn to get cross-validated MAE and R² for each model.

**Important:** scikit-learn's MAE scoring returns *negative* values (it maximises, so it negates the error). Negate the result to get the actual MAE.

In [ ]:
# Build and cross-validate your models
# Ask your AI: "Show me how to fit and cross-validate Linear Regression, Ridge,
# and Lasso using cross_val_score, reporting mean MAE and R² for each."



## Step 9: Compare Models

Create a comparison of all four models (baseline + three regression models). This could be:
- A table showing CV MAE and R² for each model
- A bar chart comparing the models visually
- Both!

Which model performed best? Is there a big difference between them?

In [ ]:
# Compare your models
# Ask your AI: "Create a bar chart comparing CV MAE for baseline, linear regression,
# Ridge, and Lasso, with the values labelled on each bar."



## Step 10: Interpret Coefficients

One of the strengths of linear models is **interpretability** — the coefficients tell you how much each feature contributes to the prediction.

A **negative coefficient** means higher values of that feature predict *lower* depression. A **positive coefficient** means higher values predict *higher* depression.

Create a horizontal bar chart of the coefficients from your best model. Which personality traits have the largest effect? Does this match what you'd expect from psychology?

In [ ]:
# Plot regression coefficients
# Ask your AI: "Plot the regression coefficients from my linear model as a
# horizontal bar chart, sorted by value, with negative coefficients in red
# and positive in blue."



## Step 11: Final Test Set Evaluation

**Now — and only now — unlock the test set.** Evaluate your best model on the held-out test data.

Compare the test performance to your cross-validated performance:
- If they're close → your model generalises well (no overfitting)
- If there's a big gap → your model may have overfit to the training data

Remember from Week 3: the test set is like a sealed exam. You only open it once, at the very end.

In [ ]:
# Evaluate your best model on the test set
# y_pred = best_model.predict(X_test)
# test_mae = mean_absolute_error(y_test, y_pred)
# test_r2 = r2_score(y_test, y_pred)
# print(f"Test MAE: {test_mae:.2f}")
# print(f"Test R²: {test_r2:.4f}")
# print(f"CV R² was: {cv_r2:.4f}")
# print(f"Gap: {abs(cv_r2 - test_r2):.4f}")



## Step 12: Document Your Findings

Take a moment to reflect on what you've done and learned. Edit this cell (double-click) with your answers:

**Which model performed best, and what was its CV R²?**

*Your answer here...*

**Which personality traits were the strongest predictors of depression?**

*Your answer here...*

**Does this match what you'd expect from psychological theory?**

*Your answer here...*

**Was there any overfitting (gap between CV and test performance)?**

*Your answer here...*

**What debugging challenges did you encounter? How did you solve them?**

*Your answer here...*

## Bonus Challenges

Finished early? Try one of these — see the [challenge brief](README.md#bonus-challenges) for full descriptions:

1. **Learning curves** — is your model underfitting or overfitting?
2. **Residual analysis** — are there patterns the model misses?
3. **Feature engineering** — interaction terms, polynomial features
4. **Compare DASS subscales** — do the same features predict Anxiety and Stress?
5. **Lasso regularisation path** — which features survive strong regularisation?
6. **Response time trap** — what happens if you include Q1E, Q2E, etc.?
7. **Data quality filtering** — does removing careless responders improve the model?
8. **Boston College COVID dataset** — try a completely different dataset
9. **Synthetic vs real** — compare with the Week 2 dataset